In [ ]:
!pip install pandas numpy scikit-learn rank_bm25 --quiet

import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

from rank_bm25 import BM25Okapi

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving dataset[1].csv to dataset[1].csv


In [ ]:
import pandas as pd

df = pd.read_csv('dataset[1].csv')

print(df.shape)
df.head()

(114000, 21)


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [ ]:
# -------------------------
# 2. CLEAN DATA
# -------------------------
df.dropna(subset=['track_name', 'artists'], inplace=True)
df.drop_duplicates(subset='track_id', inplace=True)
df.reset_index(drop=True, inplace=True)


In [ ]:
# -------------------------
# 3. CREATE COMBINED TEXT (IMPORTANT)
# Give more weight to genre
# -------------------------
df['combined_text'] = (df['track_name'].fillna('') + ' ' +
                        df['artists'].fillna('') + ' ' +
                        df['track_genre'].fillna(''))



In [ ]:
print('Cleaned dataset shape:', df.shape)
print('\nSample combined text:')
print(df['combined_text'].head(3))


Cleaned dataset shape: (89740, 22)

Sample combined text:
0                       Comedy Gen Hoshino acoustic
1            Ghost - Acoustic Ben Woodward acoustic
2    To Begin Again Ingrid Michaelson;ZAYN acoustic
Name: combined_text, dtype: object


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
# Step 5: Build TF-IDF matrix
# Use 20,000 songs for fast execution in Colab
sample_df = df.sample(20000, random_state=42).reset_index(drop=True)

# TfidfVectorizer converts text to TF-IDF feature matrix
# max_features=10000 keeps the 10,000 most common terms
# ngram_range=(1,2) captures single words AND two-word phrases
vectorizer = TfidfVectorizer(
    stop_words='english',   # remove common words like 'the', 'and'
    max_features=10000,
    ngram_range=(1, 2)      # use unigrams and bigrams
)

tfidf_matrix = vectorizer.fit_transform(sample_df['combined_text'])
print('TF-IDF matrix created successfully!')
print(f'Matrix shape: {tfidf_matrix.shape}')  # (20000 songs, 10000 terms)


TF-IDF matrix created successfully!
Matrix shape: (20000, 10000)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
# Step 6: Define the TF-IDF search function
def tfidf_search(query, top_n=5):
    # Convert query to a TF-IDF vector
    query_vec = vectorizer.transform([query])

    # Compute cosine similarity between query and all songs
    similarity = cosine_similarity(query_vec, tfidf_matrix).flatten()

    # Add similarity scores to the dataframe
    results = sample_df.copy()
    results['score'] = similarity

    # Sort by score (highest first) and return top N
    results = results.sort_values(by='score', ascending=False)
    return results[['track_name', 'artists', 'track_genre', 'score']].head(top_n)

# Step 7: Test the search function
query = input('\nEnter song to search: ')
results = tfidf_search(query)
print('\nTop Results (TF-IDF):')
print(results.to_string(index=False))



Enter song to search: acoustic love

Top Results (TF-IDF):
        track_name           artists track_genre    score
The Love Me or Die    C.W. Stoneking    acoustic 0.742444
          ハンキーパンキー       Hanare Gumi    acoustic 0.741223
              深夜高速   Kazuyoshi Saito    acoustic 0.741223
  So Are You To Me Eastmountainsouth    acoustic 0.741223
              3636            Aimyon    acoustic 0.741223


In [ ]:
# Step 1: Install BM25 library
!pip install rank_bm25 --quiet
from rank_bm25 import BM25Okapi
import re

# Step 2: Define text preprocessing function for BM25
def preprocess_for_bm25(text):
    # Convert to lowercase
    text = str(text).lower()
    # Remove special characters and punctuation
    text = re.sub(r'[^\w\s]', '', text)
    # Split into individual word tokens
    tokens = text.split()
    # Remove very short tokens (single letters)
    tokens = [t for t in tokens if len(t) > 1]
    return tokens

# Apply preprocessing to all songs in the sample
tokenized_corpus = [preprocess_for_bm25(text)
                    for text in sample_df['combined_text']]

print('Text preprocessing for BM25 complete.')
print(f'Sample tokens: {tokenized_corpus[0]}')


Text preprocessing for BM25 complete.
Sample tokens: ['feliz', 'cumpleaños', 'ferxxo', 'feid', 'latin']


In [ ]:
# Step 3: Initialise the BM25 model with the tokenised corpus
bm25 = BM25Okapi(tokenized_corpus)
print('BM25 model created successfully!\n')


BM25 model created successfully!



In [ ]:
# Step 4: Define BM25 search function
def bm25_search(query, top_n=5):
    # Tokenise the query using the same preprocessing
    tokenized_query = preprocess_for_bm25(query)

    # Get BM25 relevance scores for all documents
    scores = bm25.get_scores(tokenized_query)

    # Add scores to the dataframe
    results = sample_df.copy()
    results['score'] = scores

    # Sort by score and filter out irrelevant results (score <= 0)
    results = results[results['score'] > 0]
    results = results.sort_values(by='score', ascending=False)
    return results[['track_name', 'artists', 'track_genre', 'score']].head(top_n)

# Step 5: Test BM25 search
query = input('Enter product to search: ')
results = bm25_search(query)
print('\nTop Results (BM25):')
print(results.to_string(index=False))


Enter product to search: rock guitar

Top Results (BM25):
                        track_name               artists track_genre    score
           SMASH! - feat. PnB Rock XXXTENTACION;PnB Rock         emo 6.750472
We Will Rock You - Remastered 2011                 Queen        rock 6.434177
                           Records                Weezer        rock 6.402175
                        Revolution              Bastille        rock 6.402175
                           Pompeii              Bastille        rock 6.402175


In [ ]:
from sklearn.preprocessing import MinMaxScaler
# Content-Based Recommender using Audio Feature Vectors

# Step 1: Define audio features to use
AUDIO_FEATURES = ['danceability', 'energy', 'acousticness',
                   'valence', 'tempo', 'instrumentalness', 'speechiness']

# Step 2: Normalise all features to 0-1 range
# This prevents tempo (0-200 BPM) from dominating over
# other features that are already on a 0-1 scale
scaler = MinMaxScaler()
sample_df[AUDIO_FEATURES] = scaler.fit_transform(sample_df[AUDIO_FEATURES])

# Step 3: Define the recommendation function
def content_recommend(liked_songs, top_k=5):
    liked = sample_df[sample_df['track_name'].isin(liked_songs)]
    if liked.empty:
        print('Songs not found. Try different names.')
        return
    # Build user profile as average of liked songs' feature vectors
    user_profile = liked[AUDIO_FEATURES].mean().values.reshape(1, -1)
    # Compute cosine similarity against all songs
    scores = cosine_similarity(user_profile, sample_df[AUDIO_FEATURES].values).flatten()
    results = sample_df.copy()
    results['score'] = scores
    # Remove songs user already liked
    results = results[~results['track_name'].isin(liked_songs)]
    results = results.sort_values(by='score', ascending=False)
    return results[['track_name', 'artists', 'track_genre', 'score']].head(top_k)

# Step 4: Test the recommender
print(content_recommend(['Blinding Lights', 'Levitating'], top_k=5))


                       track_name                         artists track_genre  \
13318  Haru ni yuraredo kimi omou                   kobasolo;kopi       anime   
14738                 Twist Me Up                    Audio Bullys   breakbeat   
3370             De Vloer Is Lava       DJ Maurice;Snollebollekes   hardstyle   
12167               Tlatelolco 68                    Banda Bostik       metal   
4946         Ok Jaanu Title Track  A.R. Rahman;Srinidhi Venkatesh    pop-film   

          score  
13318  0.999715  
14738  0.999676  
3370   0.999662  
12167  0.999523  
4946   0.999521  
